In [28]:
import os, torch, shutil, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from tqdm import tqdm

# ============================================================
# 1. ROBUST SETTINGS
# ============================================================
BASE_PATH = Path("/kaggle/input/datasets/mariamhany44/100words-not-cleaned-not-preprocessed/splits")
CSV_PATH = BASE_PATH / "splits.csv"
MODEL_PATH = Path("/kaggle/input/datasets/mariamhany44/mytcnmodel/best_model (2).pth")

OUT_DIR = Path("/kaggle/working/clean_dataset")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- PARAMETERS ---
FINAL_MIN_COUNT = 20      # Minimum files to consider a "Clean" class
STRICT_NOISE_RATIO = 0.12 # Clusters smaller than this become '_noise_variant'
SIL_VALIDATION = 0.18     # Strong barrier: Only split if groups are truly distinct
OUTLIER_PRUNE = 0.05      # Prune the worst 6% of samples from EVERY cluster
# ============================================================

# ============================================================
# 2. MODEL ARCHITECTURE
# ============================================================
class SEBlock(torch.nn.Module):
    def __init__(self, c):
        super().__init__()
        self.fc = torch.nn.Sequential(
            torch.nn.Linear(c, c // 16), torch.nn.SiLU(),
            torch.nn.Linear(c // 16, c), torch.nn.Sigmoid()
        )
    def forward(self, x): return x * self.fc(x.mean(dim=2)).unsqueeze(2)

class TemporalBlock(torch.nn.Module):
    def __init__(self, ic, oc, d):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Conv1d(ic, oc, 3, padding=d, dilation=d), torch.nn.BatchNorm1d(oc), torch.nn.SiLU(),
            torch.nn.Conv1d(oc, oc, 3, padding=d, dilation=d), torch.nn.BatchNorm1d(oc), torch.nn.SiLU()
        )
        self.res = torch.nn.Conv1d(ic, oc, 1) if ic != oc else torch.nn.Identity()
        self.se = SEBlock(oc)
    def forward(self, x): return self.se(self.net(x) + self.res(x))

class FeatureModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.tcn = torch.nn.Sequential(
            TemporalBlock(438, 128, 1), TemporalBlock(128, 256, 2),
            TemporalBlock(256, 512, 4), TemporalBlock(512, 1024, 8)
        )
        self.pool = torch.nn.AdaptiveAvgPool1d(1)
    def forward(self, x): return self.pool(self.tcn(x)).squeeze(-1)

# ============================================================
# 3. INITIALIZATION
# ============================================================
print("Loading Robust Feature Extractor...")
model = FeatureModel().to(DEVICE).eval()
state = torch.load(MODEL_PATH, map_location="cpu")
model.load_state_dict({k.replace("module.", ""): v for k, v in state.items()}, strict=False)

df = pd.read_csv(CSV_PATH)
df['filepath'] = df['filepath'].str.replace('\\', '/', regex=False)
df['full_path'] = df['filepath'].apply(lambda x: BASE_PATH / x)

class_groups = df.groupby('label')
logs = []

# ============================================================
# 4. ROBUST PROCESSING
# ============================================================
for label, group in tqdm(class_groups):
    paths = [p for p in group['full_path'].tolist() if p.exists()]
    if len(paths) < FINAL_MIN_COUNT: continue

    # Extract Embeddings
    embs = []
    with torch.no_grad():
        for p in paths:
            data = np.load(p)
            if data.shape[0] != 438: data = data.T
            x = torch.tensor(data, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            embs.append(model(x).cpu().numpy().squeeze())
    
    embs = np.array(embs)
    embs = embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-8)

    # 1. Dual-Check Split Logic (K-Means + Silhouette Validation)
    best_k, labels = 1, np.zeros(len(paths))
    for k in [2, 3]:
        if len(paths) < k * FINAL_MIN_COUNT: break
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        curr_lbls = km.fit_predict(embs)
        try:
            sil = silhouette_score(embs, curr_lbls)
            # Validation Gate: Only split if separation is high-quality
            if sil > SIL_VALIDATION or (sil > 0.15 and k == 2):
                best_k, labels = k, curr_lbls
        except: continue

    to_save_data = [] # List of (file_list, folder_suffix)

    # 2. Process Chosen Clusters
    for k in range(best_k):
        c_idx = [i for i, l in enumerate(labels) if l == k]
        c_paths = [paths[i] for i in c_idx]
        c_embs = embs[c_idx]

        # Determine Suffix (Main Variant vs. Noise Variant)
        is_small_cluster = (best_k > 1) and (len(c_paths) < (len(paths) * STRICT_NOISE_RATIO))
        if is_small_cluster:
            suffix = f"_noise_v{k+1}"
        else:
            suffix = f"_v{k+1}" if best_k > 1 else ""

        # 3. Intra-Cluster Pruning (Universal Clean)
        centroid = c_embs.mean(axis=0)
        dist = np.linalg.norm(c_embs - centroid, axis=1)
        thresh = np.percentile(dist, 100 * (1 - OUTLIER_PRUNE))
        
        pruned_paths = [c_paths[i] for i in range(len(c_paths)) if dist[i] <= thresh]
        
        # Preservation Check
        if len(pruned_paths) >= 10: # Minimum samples to keep a folder alive
            to_save_data.append((pruned_paths, suffix))

    # 4. Salvage if cleaning was too destructive
    if not to_save_data:
        to_save_data = [(paths, "_unprocessed_fallback")]

    # Write to Disk
    for files, suffix in to_save_data:
        target = OUT_DIR / f"{label}{suffix}"
        target.mkdir(parents=True, exist_ok=True)
        for f in files:
            shutil.copy(f, target / f.name)
        logs.append({"class": label, "variant": suffix or "clean", "count": len(files)})

# ============================================================
# 5. EXPORT
# ============================================================
pd.DataFrame(logs).to_csv("/kaggle/working/cleaning_report.csv", index=False)

print("Finalising Zips...")
with zipfile.ZipFile("/kaggle/working/clean_dataset.zip", "w") as z:
    for f in OUT_DIR.rglob("*.npy"):
        z.write(f, f.relative_to(OUT_DIR))

print("DONE. Robust cleaning and splitting finished.")

Loading Robust Feature Extractor...


100%|██████████| 102/102 [00:17<00:00,  5.81it/s]


Finalising Zips...
DONE. Robust cleaning and splitting finished.


In [27]:
import shutil
import os

work_dir = "/kaggle/working"

for item in os.listdir(work_dir):
    path = os.path.join(work_dir, item)
    if os.path.isfile(path) or os.path.islink(path):
        os.remove(path)
    else:
        shutil.rmtree(path)

In [29]:
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

# ============================================================
# 🔥 DATA PATH (CLEANED DATASET)
# ============================================================
DATA_DIR = Path("/kaggle/working/clean_dataset")

# ============================================================
# LABEL = FOLDER NAME (IMPORTANT AFTER CLEANING)
# ============================================================
def get_label(file_path):
    return file_path.parent.name  # ✅ correct now

# ============================================================
# LOAD FILES
# ============================================================
all_files = list(DATA_DIR.rglob("*.npy"))

files = []
labels = []

print("Found files:", len(all_files))

for f in all_files:
    if "_mask.npy" in f.name:
        continue
    files.append(f)
    labels.append(get_label(f))

print("✅ Dataset loaded correctly")

# ============================================================
# BASIC STATS
# ============================================================
total_samples = len(files)
label_counts = Counter(labels)
total_classes = len(label_counts)

print("\n================ DATA OVERVIEW ================")
print(f"Total samples  : {total_samples}")
print(f"Total classes  : {total_classes}")

# ============================================================
# DATAFRAME
# ============================================================
df = pd.DataFrame(label_counts.items(), columns=["class", "count"])
df = df.sort_values("count", ascending=False).reset_index(drop=True)

# ============================================================
# THRESHOLD ANALYSIS (NOW JUST FOR INFO)
# ============================================================
THRESHOLD = 29

classes_ge = df[df["count"] >= THRESHOLD]
classes_lt = df[df["count"] < THRESHOLD]

print("\n================ THRESHOLD ANALYSIS ================")
print(f"Classes >= {THRESHOLD}: {len(classes_ge)}")
print(f"Classes <  {THRESHOLD}: {len(classes_lt)}")

print("\n% kept classes:", round(len(classes_ge)/total_classes * 100, 2), "%")
print("% removed classes:", round(len(classes_lt)/total_classes * 100, 2), "%")

# ============================================================
# INSIGHTS
# ============================================================
print("\n================ INSIGHTS ================")
print(f"Min samples per class: {df['count'].min()}")
print(f"Max samples per class: {df['count'].max()}")
print(f"Mean samples per class: {df['count'].mean():.2f}")
print(f"Median samples per class: {df['count'].median():.2f}")

# ============================================================
# TOP / BOTTOM
# ============================================================
print("\n================ TOP 10 ================")
print(df.head(10))

print("\n================ BOTTOM 10 ================")
print(df.tail(10))

# ============================================================
# SAVE
# ============================================================
df.to_csv("/kaggle/working/clean_class_distribution.csv", index=False)

print("\n📁 Saved: clean_class_distribution.csv")

Found files: 3068
✅ Dataset loaded correctly

================ DATA OVERVIEW ================
Total samples  : 3068
Total classes  : 106

================ THRESHOLD ANALYSIS ================
Classes >= 29: 65
Classes <  29: 41

% kept classes: 61.32 %
% removed classes: 38.68 %

================ INSIGHTS ================
Min samples per class: 19
Max samples per class: 58
Mean samples per class: 28.94
Median samples per class: 29.00

================ TOP 10 ================
       class  count
0      ABOUT     58
1  AUTISM_v2     40
2     ARM_v2     33
3     ARM_v1     31
4  ARREST_v1     31
5  5 DOLLARS     30
6      AFTER     30
7   ALPHABET     30
8  ADVERTISE     30
9     APPEAR     30

================ BOTTOM 10 ================
          class  count
96      AMAZING     27
97        AGAIN     27
98       ARRIVE     27
99        ADULT     27
100   AGREEMENT     27
101    ACCIDENT     27
102  ACCOMPLISH     27
103   ARREST_v2     26
104      ARM_v3     21
105   AUTISM_v1     19

📁 

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path
from tqdm import tqdm
import random
from collections import Counter
import joblib

# ============================================================
# CONFIG
# ============================================================
DATA_DIR = Path("/kaggle/working/clean_dataset")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_FRAMES, FEATURE_DIM = 158, 438
BATCH_SIZE, EPOCHS = 64, 60
MAX_LR, WEIGHT_DECAY, PATIENCE = 6e-4, 0.12, 15
MODEL_SAVE_PATH, LE_SAVE_PATH = "best_asl_model.pth", "label_encoder.pkl"

# ============================================================
# DATASET & MIXUP (Same as previous refined version)
# ============================================================
def augment_landmarks(x):
    if random.random() < 0.5: # Time stretch
        speed = random.uniform(0.9, 1.1)
        new_len = max(16, int(x.shape[1] * speed))
        x = nn.functional.interpolate(x.unsqueeze(0), size=new_len, mode='linear', align_corners=False).squeeze(0)
    if random.random() < 0.3: x = x + torch.randn_like(x) * 0.0008 # Jitter
    return nn.functional.pad(x[:, :TARGET_FRAMES], (0, max(0, TARGET_FRAMES - x.shape[1])))

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(DEVICE)
    return lam * x + (1 - lam) * x[index, :], y, y[index], lam

class ASLDataset(Dataset):
    def __init__(self, files, labels, train=True):
        self.files, self.labels, self.train = files, labels, train
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        x = np.load(self.files[idx])
        if x.shape[0] != FEATURE_DIM: x = x.T
        x = torch.from_numpy(x).float()
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        x = augment_landmarks(x) if self.train else nn.functional.pad(x[:, :TARGET_FRAMES], (0, max(0, TARGET_FRAMES - x.shape[1])))
        return x, y

# ============================================================
# MODEL: ATTENTION-AUGMENTED TCN
# ============================================================
class SimplifiedAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1), nn.Softmax(dim=1))
    def forward(self, x):
        # x: [B, C, T] -> [B, T, C]
        x_t = x.transpose(1, 2)
        weights = self.attn(x_t)
        return torch.sum(x_t * weights, dim=1)

class TemporalBlock(nn.Module):
    def __init__(self, ic, oc, d, drop=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(ic, oc, 3, padding=d, dilation=d), nn.BatchNorm1d(oc), nn.Mish(), nn.Dropout1d(drop),
            nn.Conv1d(oc, oc, 3, padding=d, dilation=d), nn.BatchNorm1d(oc), nn.Mish()
        )
        self.res = nn.Conv1d(ic, oc, 1) if ic != oc else nn.Identity()
    def forward(self, x): return nn.functional.mish(self.net(x) + self.res(x))

class StrongASLModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(FEATURE_DIM, 256), nn.Mish(), nn.Dropout(0.2))
        self.tcn = nn.Sequential(
            TemporalBlock(256, 256, 1), TemporalBlock(256, 512, 2),
            TemporalBlock(512, 512, 4), TemporalBlock(512, 1024, 8)
        )
        self.attn_pool = SimplifiedAttention(1024)
        self.fc = nn.Sequential(
            nn.Linear(1024, 1024), nn.Mish(), nn.Dropout(0.5),
            nn.Linear(1024, num_classes)
        )
    def forward(self, x):
        # x: [B, 438, T]
        x = self.stem(x.transpose(1, 2)).transpose(1, 2)
        x = self.tcn(x)
        x = self.attn_pool(x)
        return self.fc(x)

# ============================================================
# TRAIN SCRIPT (Standard Logic)
# ============================================================
all_files = [f for f in DATA_DIR.rglob("*.npy") if "_mask.npy" not in f.name]
labels_raw = [f.parent.name for f in all_files]
counts = Counter(labels_raw)
top_set = set(sorted([c for c in counts if counts[c] >= 30], key=lambda x: counts[x], reverse=True)[:1000])

files, labels = zip(*[(str(f), l) for f, l in zip(all_files, labels_raw) if l in top_set])
le = LabelEncoder()
y = le.fit_transform(labels)
num_classes = len(le.classes_)

X_train, X_tmp, y_train, y_tmp = train_test_split(files, y, test_size=0.15, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

train_loader = DataLoader(ASLDataset(X_train, y_train, True), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ASLDataset(X_val, y_val, False), batch_size=BATCH_SIZE)
test_loader = DataLoader(ASLDataset(X_test, y_test, False), batch_size=BATCH_SIZE)

model = StrongASLModel(num_classes).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
scheduler = OneCycleLR(optimizer, max_lr=MAX_LR, epochs=EPOCHS, steps_per_epoch=len(train_loader))
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler()

best_acc, patience_counter = 0, 0

for epoch in range(EPOCHS):
    model.train()
    tl, tc, tt = 0, 0, 0
    for x, yb in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        x, yb = x.to(DEVICE), yb.to(DEVICE)
        x, y_a, y_b, lam = mixup_data(x, yb)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out = model(x)
            loss = lam * criterion(out, y_a) + (1 - lam) * criterion(out, y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tl += loss.item()
        tc += (lam * (out.argmax(1) == y_a).float() + (1 - lam) * (out.argmax(1) == y_b).float()).sum().item()
        tt += yb.size(0)

    model.eval()
    vc, vt = 0, 0
    with torch.no_grad():
        for x, yb in val_loader:
            x, yb = x.to(DEVICE), yb.to(DEVICE)
            out = model(x)
            vc += (out.argmax(1) == yb).sum().item(); vt += yb.size(0)

    print(f"Epoch {epoch+1}: Train Acc {tc/tt:.4f} | Val Acc {vc/vt:.4f}")
    if vc/vt > best_acc:
        best_acc, patience_counter = vc/vt, 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH); joblib.dump(le, LE_SAVE_PATH)
    elif (patience_counter := patience_counter + 1) >= PATIENCE: break

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval(); preds, gts = [], []
for x, yb in test_loader:
    out = model(x.to(DEVICE))
    preds.extend(out.argmax(1).cpu().numpy()); gts.extend(yb.numpy())
print(f"\nFINAL TEST ACC: {np.mean(np.array(preds) == np.array(gts)):.4f}")

Epoch 1: 100%|██████████| 515/515 [00:59<00:00,  8.64it/s]


Epoch 1: Train Acc 0.0057 | Val Acc 0.0275


Epoch 2: 100%|██████████| 515/515 [00:59<00:00,  8.69it/s]


Epoch 2: Train Acc 0.0210 | Val Acc 0.0506


Epoch 3: 100%|██████████| 515/515 [00:59<00:00,  8.62it/s]


Epoch 3: Train Acc 0.0412 | Val Acc 0.0877


Epoch 4:  80%|████████  | 414/515 [00:47<00:11,  8.50it/s]